# Final Individual Project: Screen Time and Sleep in the 2007 YRBS

**Research question:** Among U.S. high school students in the 2007 YRBS, is recreational computer / video game use associated with average sleep duration on school nights?

This notebook follows a complete statistical workflow: research question → data preparation → method → result → interpretation.


## 1. Import packages and set paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Make paths work whether this notebook is launched from the project root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'raw' / 'YRBS_2007.csv'
OUT_DATA = ROOT / 'data' / 'processed'
OUT_FIG = ROOT / 'output' / 'figures'
OUT_TABLE = ROOT / 'output' / 'tables'
for p in [OUT_DATA, OUT_FIG, OUT_TABLE]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 130
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 2. Load data

In [ ]:
df = pd.read_csv(DATA_RAW)
print('Raw dataset shape:', df.shape)
df[['Sleep', 'ComputerUse', 'WhatIsYourSex', 'InWhatGradeAreYou']].head()

## 3. Define variables and clean data

Selected variables:

- `Sleep`: average sleep on school nights.
- `ComputerUse`: video/computer games or non-school computer use on an average school day.
- `WhatIsYourSex`: sex.
- `InWhatGradeAreYou`: grade.

Rows with missing values in selected variables are removed. `Ungraded or other grade` is removed because it is not part of the standard grade comparison and has very small frequency.


In [ ]:
selected_cols = ['Sleep', 'ComputerUse', 'WhatIsYourSex', 'InWhatGradeAreYou', 'weight', 'stratum', 'psu']
clean = df[selected_cols].copy()

missing_before = clean.isna().sum().to_frame('missing_n')
missing_before['missing_pct'] = (clean.isna().mean() * 100).round(2)
missing_before

In [ ]:
clean = clean.dropna(subset=['Sleep', 'ComputerUse', 'WhatIsYourSex', 'InWhatGradeAreYou'])
clean = clean[clean['InWhatGradeAreYou'].isin([1, 2, 3, 4])].copy()

sleep_map = {1: 4, 2: 5, 3: 6, 4: 7, 5: 8, 6: 9, 7: 10}
sleep_label_map = {1: '4 or less hours', 2: '5 hours', 3: '6 hours', 4: '7 hours', 5: '8 hours', 6: '9 hours', 7: '10 or more hours'}
computer_hours_map = {1: 0, 2: 0.5, 3: 1, 4: 2, 5: 3, 6: 4, 7: 5}
computer_label_map = {1: '0 h', 2: '<1 h', 3: '1 h', 4: '2 h', 5: '3 h', 6: '4 h', 7: '5+ h'}
sex_map = {1: 'Female', 2: 'Male'}
grade_map = {1: '9th', 2: '10th', 3: '11th', 4: '12th'}

clean['sleep_hours_approx'] = clean['Sleep'].astype(int).map(sleep_map)
clean['sleep_label'] = clean['Sleep'].astype(int).map(sleep_label_map)
clean['computer_hours_approx'] = clean['ComputerUse'].astype(int).map(computer_hours_map)
clean['computer_use_label'] = clean['ComputerUse'].astype(int).map(computer_label_map)
clean['sex'] = clean['WhatIsYourSex'].astype(int).map(sex_map)
clean['grade'] = clean['InWhatGradeAreYou'].astype(int).map(grade_map)
clean['sleep_8plus'] = (clean['sleep_hours_approx'] >= 8).astype(int)

computer_order = ['0 h', '<1 h', '1 h', '2 h', '3 h', '4 h', '5+ h']
grade_order = ['9th', '10th', '11th', '12th']
clean['computer_use_label'] = pd.Categorical(clean['computer_use_label'], categories=computer_order, ordered=True)
clean['grade'] = pd.Categorical(clean['grade'], categories=grade_order, ordered=True)

clean_out = clean[['sleep_hours_approx', 'sleep_label', 'computer_hours_approx', 'computer_use_label', 'sex', 'grade', 'sleep_8plus', 'weight', 'stratum', 'psu']].copy()
clean_out.to_csv(OUT_DATA / 'yrbs_2007_screen_sleep_cleaned.csv', index=False)

print('Cleaned dataset shape:', clean_out.shape)
clean_out.head()

## 4. Descriptive statistics

In [ ]:
summary = clean_out.groupby('computer_use_label', observed=False).agg(
    n=('sleep_hours_approx', 'size'),
    mean_sleep_hours=('sleep_hours_approx', 'mean'),
    sd_sleep_hours=('sleep_hours_approx', 'std'),
    se_sleep_hours=('sleep_hours_approx', lambda x: x.std(ddof=1) / np.sqrt(x.count())),
    pct_8plus_sleep=('sleep_8plus', lambda x: 100 * x.mean())
).reset_index()
summary['ci95_low'] = summary['mean_sleep_hours'] - 1.96 * summary['se_sleep_hours']
summary['ci95_high'] = summary['mean_sleep_hours'] + 1.96 * summary['se_sleep_hours']
summary.to_csv(OUT_TABLE / 'group_summary_by_computer_use.csv', index=False)
summary

## 5. Figure 1: Mean sleep by screen-time group

In [ ]:
x = np.arange(len(summary))
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.bar(x, summary['mean_sleep_hours'])
ax.errorbar(x, summary['mean_sleep_hours'], yerr=1.96 * summary['se_sleep_hours'], fmt='none', capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(summary['computer_use_label'].astype(str))
ax.set_ylabel('Approximate sleep hours')
ax.set_xlabel('Non-school computer / video game use on an average school day')
ax.set_title('Average sleep hours by recreational screen time')
ax.set_ylim(0, 8)
ax.grid(axis='y', alpha=0.3)
for i, val in enumerate(summary['mean_sleep_hours']):
    ax.text(i, val + 0.12, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_FIG / 'fig1_mean_sleep_by_screen_time.png', bbox_inches='tight')
plt.show()

## 6. Method 1: One-way ANOVA

Null hypothesis: mean sleep hours are the same across all computer-use groups.


In [ ]:
anova_model = smf.ols('sleep_hours_approx ~ C(computer_use_label)', data=clean_out).fit()
anova_tbl = anova_lm(anova_model, typ=2)
eta2 = anova_tbl.loc['C(computer_use_label)', 'sum_sq'] / anova_tbl['sum_sq'].sum()
anova_tbl['eta_squared'] = [eta2, np.nan]
anova_tbl.to_csv(OUT_TABLE / 'anova_sleep_by_computer_use.csv')
anova_tbl

In [ ]:
tukey = pairwise_tukeyhsd(endog=clean_out['sleep_hours_approx'], groups=clean_out['computer_use_label'], alpha=0.05)
print(tukey.summary())

## 7. Figure 2: Percent sleeping 8+ hours

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(x, summary['pct_8plus_sleep'], marker='o')
ax.set_xticks(x)
ax.set_xticklabels(summary['computer_use_label'].astype(str))
ax.set_ylabel('Students sleeping 8+ hours (%)')
ax.set_xlabel('Non-school computer / video game use on an average school day')
ax.set_title('Share of students getting 8+ hours of sleep')
ax.set_ylim(0, 40)
ax.grid(axis='y', alpha=0.3)
for i, val in enumerate(summary['pct_8plus_sleep']):
    ax.text(i, val + 0.9, f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_FIG / 'fig2_pct_8plus_sleep_by_screen_time.png', bbox_inches='tight')
plt.show()

## 8. Method 2: Regression

Model: approximate sleep hours = computer-use hours + sex + grade.

HC3 robust standard errors are used because the outcome is based on ordinal survey categories converted into approximate hours.


In [ ]:
reg_model = smf.ols('sleep_hours_approx ~ computer_hours_approx + C(sex) + C(grade)', data=clean_out).fit(cov_type='HC3')
print(reg_model.summary())

regression_table = pd.DataFrame({
    'term': reg_model.params.index,
    'coef': reg_model.params.values,
    'std_error_HC3': reg_model.bse.values,
    'z_value': reg_model.tvalues.values,
    'p_value': reg_model.pvalues.values,
    'ci95_low': reg_model.conf_int()[0].values,
    'ci95_high': reg_model.conf_int()[1].values,
})
regression_table.to_csv(OUT_TABLE / 'regression_sleep_screen_time.csv', index=False)
regression_table

## 9. Figure 3: Regression visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
bin_means = clean_out.groupby('computer_hours_approx', observed=False).agg(mean_sleep=('sleep_hours_approx', 'mean'), n=('sleep_hours_approx', 'size')).reset_index()
ax.scatter(clean_out['computer_hours_approx'], clean_out['sleep_hours_approx'], s=7, alpha=0.08)
ax.plot(bin_means['computer_hours_approx'], bin_means['mean_sleep'], marker='o', linewidth=2, label='Group mean')
vis_model = smf.ols('sleep_hours_approx ~ computer_hours_approx', data=clean_out).fit()
x_line = np.linspace(0, 5, 100)
y_line = vis_model.params['Intercept'] + vis_model.params['computer_hours_approx'] * x_line
ax.plot(x_line, y_line, linestyle='--', linewidth=2, label='Unadjusted fitted line')
ax.set_xlabel('Approximate non-school computer / video game hours')
ax.set_ylabel('Approximate sleep hours')
ax.set_title('Screen time and school-night sleep')
ax.set_yticks([4, 5, 6, 7, 8, 9, 10])
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / 'fig3_regression_screen_sleep.png', bbox_inches='tight')
plt.show()

## 10. Interpretation and conclusion

The cleaned sample includes **12,042** students.

The ANOVA result shows that mean sleep hours differ across recreational computer-use groups: **F(6, 12035) = 17.41, p = 3.61e-20**. The eta-squared value is **0.009**, which indicates that the group differences are statistically significant but small in practical size.

The regression result shows that each additional approximate hour of recreational computer/video game use is associated with about **5.6 fewer minutes** of school-night sleep after adjusting for sex and grade. The coefficient is statistically significant, but the model R² is **0.039**, so screen time explains only a small portion of the variation in sleep.

**Conclusion:** Students with higher recreational screen time tend to report slightly shorter sleep, but the association is small. Since this is observational survey data, the result should be interpreted as an association rather than causal evidence.
